In [3]:
import numpy as np
import random

# ============================================================
#           Q-LEARNING FROM SCRATCH
# ============================================================
# Le Q-Learning est un algorithme de Reinforcement Learning.
#
# L’idée :
# Un agent apprend tout seul à prendre les bonnes décisions en interagissant avec un environnement.
#
# L’agent :
# - observe un état (state)
# - choisit une action
# - reçoit une récompense (reward)
# - apprend progressivement la meilleure stratégie
#
# Exemple réel :
# ----------------
# Un robot doit trouver la sortie d’un labyrinthe.
#
# - Bonne direction  -> récompense positive
# - Mauvaise direction -> pénalité
#
# Après plusieurs essais :
# le robot apprend le meilleur chemin.
#
# ============================================================
#                ENVIRONNEMENT
# ============================================================

# Grille 4x4
#
# S = Start (départ)
# G = Goal  (objectif)
#
# 0  1  2  3
# 4  5  6  7
# 8  9 10 11
#12 13 14 15
#
# Objectif :
# atteindre la case 15

N_STATES = 16

# Actions possibles
#
# 0 = haut
# 1 = bas
# 2 = gauche
# 3 = droite

N_ACTIONS = 4

GOAL_STATE = 15

# ============================================================
#              HYPERPARAMÈTRES
# ============================================================

# Taux d’apprentissage
#
# α (alpha)
#
# Contrôle :
# combien l’agent apprend des nouvelles expériences
#
# petit alpha  -> apprentissage lent
# grand alpha  -> apprentissage rapide

LEARNING_RATE = 0.1

# Facteur de réduction
#
# γ (gamma)
#
# Importance des récompenses futures
#
# proche de 1 :
# l’agent pense au futur
#
# proche de 0 :
# l’agent pense seulement au présent

DISCOUNT_FACTOR = 0.9

# Epsilon :
# probabilité d’exploration
#
# epsilon grand :
# beaucoup d’exploration
#
# epsilon petit :
# exploitation des connaissances

EPSILON = 0.2

# Nombre d’épisodes d’entraînement

EPISODES = 1000

# ============================================================
#               INITIALISATION Q-TABLE
# ============================================================

# La Q-table contient :
#
# Q(state, action)
#
# Elle mémorise :
# "à quel point une action est bonne dans un état"
#
# Dimensions :
# 16 états × 4 actions

Q_table = np.zeros((N_STATES, N_ACTIONS))

# ============================================================
#           FONCTION DE TRANSITION
# ============================================================

def get_next_state(state, action):

    """
    Cette fonction calcule :
    - le prochain état
    - la récompense
    """

    row = state // 4
    col = state % 4

    # =========================================
    # Déplacement
    # =========================================

    # Haut
    if action == 0 and row > 0:
        row -= 1

    # Bas
    elif action == 1 and row < 3:
        row += 1

    # Gauche
    elif action == 2 and col > 0:
        col -= 1

    # Droite
    elif action == 3 and col < 3:
        col += 1

    next_state = row * 4 + col

    # =========================================
    # Récompenses
    # =========================================

    # Si objectif atteint
    if next_state == GOAL_STATE:
        reward = 100

    # Sinon petite pénalité
    else:
        reward = -1

    return next_state, reward

# ============================================================
#                CHOIX D’ACTION
# ============================================================

def choose_action(state):

    """
    Politique epsilon-greedy
    -------------------------
    Deux comportements :

    1. Exploration :
       essayer des actions aléatoires

    2. Exploitation :
       utiliser la meilleure action connue
    """

    # Exploration
    if random.uniform(0, 1) < EPSILON:

        action = random.randint(0, N_ACTIONS - 1)

    # Exploitation
    else:

        # argmax :
        # retourne l’indice de la plus grande valeur

        action = np.argmax(Q_table[state])

    return action

# ============================================================
#                ENTRAÎNEMENT
# ============================================================

print("Début de l'entraînement...\n")

for episode in range(EPISODES):

    # L’agent commence toujours à l’état 0
    state = 0

    done = False

    while not done:

        # =====================================
        # Choisir une action
        # =====================================

        action = choose_action(state)

        # =====================================
        # Exécuter l’action
        # =====================================

        next_state, reward = get_next_state(state, action)

        # =====================================
        # FORMULE DU Q-LEARNING
        # =====================================

        # Ancienne valeur
        old_value = Q_table[state, action]

        # Meilleure valeur future
        next_max = np.max(Q_table[next_state])

        # ==================================================
        # Formule mathématique :
        # ==================================================

        # Q(s,a) ← Q(s,a) + α [ r + γ max Q(s',a') - Q(s,a) ]
        #
        # Avec :
        #
        # s  = état actuel
        # a  = action actuelle
        # r  = récompense
        # s' = prochain état
        #
        # α = learning rate
        # γ = discount factor
        #
        # ==================================================

        new_value = old_value + LEARNING_RATE * (
            reward + DISCOUNT_FACTOR * next_max - old_value
        )

        # Mise à jour de la Q-table
        Q_table[state, action] = new_value

        # Passer au nouvel état
        state = next_state

        # Vérifier si objectif atteint
        if state == GOAL_STATE:
            done = True

# ============================================================
#                 AFFICHAGE Q-TABLE
# ============================================================

print("Q-table finale :\n")

print(Q_table)

# ============================================================
#              TEST DE L’AGENT
# ============================================================

print("\nTest du chemin appris :\n")

state = 0

path = [state]

done = False

while not done:

    # Choisir la meilleure action
    action = np.argmax(Q_table[state])

    # Exécuter l’action
    next_state, reward = get_next_state(state, action)

    path.append(next_state)

    state = next_state

    # Fin
    if state == GOAL_STATE:
        done = True

print("Chemin trouvé :")
print(path)

# ============================================================
#            INTERPRÉTATION DU RÉSULTAT
# ============================================================

# Exemple :
#
# [0, 1, 2, 3, 7, 11, 15]
#
# Cela signifie :
#
# L’agent a appris que ce chemin
# est le meilleur pour atteindre le but.
#
# ============================================================

Début de l'entraînement...

Q-table finale :

[[ 48.23570286  54.9539      48.25094248  54.8373554 ]
 [ 19.85457472  62.16993061  29.23986449   9.42381791]
 [ -0.385219    46.09908412  -0.42456571  -0.2962    ]
 [ -0.385219    23.48303626  -0.30061602  -0.393967  ]
 [ 48.2156898   61.95405117  54.77328754  62.171     ]
 [ 54.28961717  70.19        54.55531599  69.77359415]
 [  9.58407192  79.09888213   6.03071     31.01671478]
 [  2.15468071  88.87505785  18.8260068   17.02371772]
 [ 33.77065357  10.8572038   37.44151724  70.18941004]
 [ 62.05097414  77.1489478   61.71715347  79.1       ]
 [ 69.23155019  88.50125596  69.66475883  89.        ]
 [ 78.11771309 100.          78.90961248  88.86809794]
 [ 42.17240751  -0.3940399   -0.392509     4.25790024]
 [ 23.94878715   6.82722222   7.33193419  88.67338693]
 [ 41.17400037  53.77172382  43.66827419  99.99806367]
 [  0.           0.           0.           0.        ]]

Test du chemin appris :

Chemin trouvé :
[0, 4, 5, 9, 10, 11, 15]
